In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../outputs/finance.db')
print("Connecté à finance.db")

# Règle 1 : montants négatifs ou nuls
q1 = pd.read_sql_query("SELECT COUNT(*) AS nb FROM fact_transactions WHERE amount <= 0", conn)

# Règle 2 : incohérences soldes
q2 = pd.read_sql_query("""
    SELECT COUNT(*) AS nb FROM fact_transactions
    WHERE ROUND(solde_avant_source - amount, 2) != ROUND(solde_apres_source, 2)
    AND solde_apres_source != 0
""", conn)

# Règle 3 : transactions frauduleuses non flagguées
q3 = pd.read_sql_query("""
    SELECT COUNT(*) AS nb FROM fact_transactions
    WHERE est_fraude = 1 AND est_flagge = 0
""", conn)

# Règle 4 : employés sans département
q4 = pd.read_sql_query("""
    SELECT COUNT(*) AS nb FROM fact_employes
    WHERE dept_id IS NULL
""", conn)

# Règle 5 : employés avec ancienneté négative
q5 = pd.read_sql_query("""
    SELECT COUNT(*) AS nb FROM fact_employes
    WHERE anciennete < 0
""", conn)

rapport_qualite = pd.DataFrame({
    'regle'     : [
        'Montants invalides',
        'Soldes incohérents',
        'Fraudes non flagguées',
        'Employés sans département',
        'Ancienneté négative'
    ],
    'nb_anomalies' : [
        q1['nb'].values[0],
        q2['nb'].values[0],
        q3['nb'].values[0],
        q4['nb'].values[0],
        q5['nb'].values[0]
    ]
})

rapport_qualite['statut'] = rapport_qualite['nb_anomalies'].apply(
    lambda x: 'OK' if x == 0 else 'ALERTE'
)

print(rapport_qualite)
rapport_qualite.to_csv('../outputs/rapport_qualite_final.csv', index=False)


kpi_transactions = pd.read_sql_query("""
    SELECT
        COUNT(*)                                        AS nb_total_transactions,
        ROUND(SUM(amount), 2)                           AS volume_total,
        ROUND(AVG(amount), 2)                           AS montant_moyen,
        SUM(est_fraude)                                 AS nb_fraudes,
        ROUND(SUM(est_fraude) * 100.0 / COUNT(*), 3)   AS taux_fraude_pct,
        COUNT(DISTINCT compte_source)                   AS nb_comptes_uniques
    FROM fact_transactions
""", conn)

print("=== KPI TRANSACTIONS ===")
print(kpi_transactions.T)
kpi_transactions.to_csv('../outputs/kpi_transactions.csv', index=False)

kpi_employes = pd.read_sql_query("""
    SELECT
        COUNT(DISTINCT employe_id)                          AS nb_employes_total,
        ROUND(AVG(age), 1)                                  AS age_moyen,
        ROUND(AVG(anciennete), 1)                           AS anciennete_moyenne,
        SUM(CASE WHEN statut = 'ACTIVE' THEN 1 ELSE 0 END)     AS nb_actifs,
        SUM(CASE WHEN statut = 'TERMINATED' THEN 1 ELSE 0 END) AS nb_departs,
        ROUND(SUM(CASE WHEN statut = 'TERMINATED' THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                AS taux_depart_pct
    FROM fact_employes
    WHERE annee = (SELECT MAX(annee) FROM fact_employes)
""", conn)

print("=== KPI EMPLOYES ===")
print(kpi_employes.T)
kpi_employes.to_csv('../outputs/kpi_employes.csv', index=False)


# Export 1 : transactions avec libellé type pour Power BI
export_transactions = pd.read_sql_query("""
    SELECT
        ft.transaction_id,
        dt.type,
        ft.step,
        ft.amount,
        ft.compte_source,
        ft.compte_dest,
        ft.solde_avant_source,
        ft.solde_apres_source,
        ft.est_fraude,
        ft.est_flagge
    FROM fact_transactions ft
    JOIN dim_type_transaction dt ON ft.type_id = dt.type_id
""", conn)
export_transactions.to_csv('../outputs/powerbi_transactions.csv', index=False)
print(f"powerbi_transactions.csv : {len(export_transactions)} lignes")

# Export 2 : employés avec département, poste, ville pour Power BI
export_employes = pd.read_sql_query("""
    SELECT
        fe.employe_id,
        dd.departement,
        dd.unite,
        dp.poste,
        dv.ville,
        fe.annee,
        fe.age,
        fe.anciennete,
        fe.genre,
        fe.statut,
        fe.raison_depart,
        fe.type_depart
    FROM fact_employes fe
    JOIN dim_departement dd ON fe.dept_id = dd.dept_id
    JOIN dim_poste dp ON fe.poste_id = dp.poste_id
    JOIN dim_ville dv ON fe.ville_id = dv.ville_id
""", conn)
export_employes.to_csv('../outputs/powerbi_employes.csv', index=False)
print(f"powerbi_employes.csv : {len(export_employes)} lignes")

conn.close()
print("\nTous les fichiers exportés dans outputs/")

Connecté à finance.db
                       regle  nb_anomalies  statut
0         Montants invalides             0      OK
1         Soldes incohérents       1524010  ALERTE
2      Fraudes non flagguées          8181  ALERTE
3  Employés sans département             0      OK
4        Ancienneté négative             0      OK
=== KPI TRANSACTIONS ===
                                  0
nb_total_transactions  6.362604e+06
volume_total           1.144393e+12
montant_moyen          1.798624e+05
nb_fraudes             8.197000e+03
taux_fraude_pct        1.290000e-01
nb_comptes_uniques     6.353291e+06
=== KPI EMPLOYES ===
                          0
nb_employes_total   4961.00
age_moyen             42.90
anciennete_moyenne    13.40
nb_actifs           4799.00
nb_departs           162.00
taux_depart_pct        3.27
powerbi_transactions.csv : 6362604 lignes
powerbi_employes.csv : 49653 lignes

Tous les fichiers exportés dans outputs/
